# N1 · 配置 + 追踪: 把消融矩阵变成可查询的实验库 (Config & Tracking)

> 配套 9.5-L2/L3 · **真实科研动作**: 用 `exp_tracker` 把 9.4 的消融矩阵完整跑一遍并自动留痕
> (config + metrics + git/env 指纹), 再用 pandas 把 jsonl 变成「可查询的实验数据库」。

复用 9.4 的 `experiment.py` 当被追踪的真实验 —— 设计(9.4) + 留痕(9.5) 在这里接上。

In [1]:
import sys
from pathlib import Path
SRC = Path.cwd().parent / "src"
EXP_SRC = Path.cwd().parent.parent / "experiment-design" / "src"   # 复用 9.4 的模拟器
for p in (SRC, EXP_SRC):
    sys.path.insert(0, str(p))
import pandas as pd
import experiment as ex      # 9.4 的确定性模拟器
import exp_tracker as et     # 本专题的本地追踪器
print("方法:", ex.METHODS, "| 噪声:", ex.NOISES)

方法: ['DPO', 'Robust-DPO'] | 噪声: [0.0, 0.2, 0.4]


## 1. config as code: 一个 run = 一份 config + 一个 seed (L2)

消融矩阵 = 对 (method × noise × seed) 做笛卡尔积, 每个组合是一份 config。
跑每个 run 时用 `exp_tracker.init` 自动盖上 git_sha + 环境指纹。

In [2]:
RUNS_DIR = Path.cwd() / "_runs"
import shutil
if RUNS_DIR.exists(): shutil.rmtree(RUNS_DIR)   # 干净重跑

for method in ex.METHODS:
    for noise in ex.NOISES:
        for seed in range(8):
            config = {"method": method, "noise": noise, "seed": seed,
                      "dataset": "sim-prefs@v1"}     # 注意: 数据也带版本 (L4)
            run = et.init("dpo-noise", config=config, out_dir=RUNS_DIR)  # 盖指纹
            win = ex.run_experiment(method, noise, seed)                 # 9.4 的实验
            run.log({"win_rate": round(win, 4)})
            run.finish()                                                # 落盘 jsonl
print("跑完并留痕:", 2*3*8, "个 run →", (RUNS_DIR/'dpo-noise.jsonl').name)

跑完并留痕: 48 个 run → dpo-noise.jsonl


## 2. 一条记录里到底存了什么 (三件套: config + metrics + 指纹)

In [3]:
import json
runs = et.load_runs("dpo-noise", out_dir=RUNS_DIR)
print("一条记录的全貌:")
print(json.dumps(runs[0], ensure_ascii=False, indent=2)[:700])

一条记录的全貌:
{
  "project": "dpo-noise",
  "config": {
    "method": "DPO",
    "noise": 0.0,
    "seed": 0,
    "dataset": "sim-prefs@v1"
  },
  "metrics": {
    "win_rate": 0.6223
  },
  "git_sha": "0ac9db7-dirty",
  "env": {
    "python": "3.13.13",
    "platform": "Windows",
    "packages": {
      "numpy": "2.4.6",
      "scipy": "1.17.1",
      "torch": "2.13.0.dev20260602+cu130",
      "transformers": "4.57.6"
    }
  }
}


## 3. jsonl → pandas: 「可查询的实验数据库」(L3)

这就是「一锅粥」和「可查询数据库」的区别: 30 个 run 不再记在脑子里/截图里, 而是一个表。

In [4]:
df = pd.json_normalize(runs)
# 按 method×noise 聚合 (这正是 9.4 的 aggregate, 现在数据来自留痕记录)
summary = (df.groupby(["config.method", "config.noise"])["metrics.win_rate"]
             .agg(["mean", "std", "count"]).round(3))
summary

mean    std  count
config.method config.noise                     
DPO           0.0           0.619  0.017      8
              0.2           0.501  0.024      8
              0.4           0.397  0.015      8
Robust-DPO    0.0           0.615  0.022      8
              0.2           0.591  0.017      8
              0.4           0.548  0.020      8

## 4. 真·查询: 「高噪声(0.4)下哪个方法最好?」—— 一行, 不用考古

In [5]:
q = df[df["config.noise"] == 0.4]
best = q.groupby("config.method")["metrics.win_rate"].mean().sort_values(ascending=False)
print("noise=0.4 下各方法平均 win_rate:")
print(best.round(3).to_string())
print(f"\n→ 赢家: {best.index[0]}  (这个问题现在是一次 query, 不是一次考古)")
print(f"→ 每条记录都带 git_sha={runs[0]['git_sha']}, 半年后仍知道是哪版代码跑的。")

noise=0.4 下各方法平均 win_rate:
config.method
Robust-DPO    0.548
DPO           0.397

→ 赢家: Robust-DPO  (这个问题现在是一次 query, 不是一次考古)
→ 每条记录都带 git_sha=0ac9db7-dirty, 半年后仍知道是哪版代码跑的。


## 5. 反思

你把 9.4 的消融矩阵从「跑完就散」升级成了「自动留痕、可查询、带版本指纹」。
注意 git_sha 可能带 `-dirty` —— 说明跑的时候代码有未提交改动, 正式结果应先 commit 让它 clean (L3)。

**真用时**: 把 `et` 换成 `import wandb`, 接口几乎一样 (`init/log/finish`)。你刚理解的「一条记录该有什么」
完全迁移。本地 jsonl 版的好处是零依赖、可 grep、可进 git diff、十年后还打得开。

下一步: 去 N2 给实验记录做「可复现性体检」, 并验证确定性。